In [1]:
#!/usr/bin/env python3
"""
Generate all parameter .in files for the Z4 domain-wall model scan.

Physics (new parameterization):
  V = -μ²|Φ|² + λ₁|Φ|⁴ − λ₂(Φ⁴ + Φ*⁴)
    = -(μ²/2)(h²+a²) + (λ₁/4)(h²+a²)² − (λ₂/2)(h⁴−6h²a²+a⁴) + V₀

  Vacuum (h-direction): v² = μ² / (λ₁ − 2λ₂)
  Unit-vev constraint (v=1):  λ₁ = μ² + 2λ₂

Sweep axes:
  mu      : [0.05, 0.10]                            – physical mass scale; H0 = H0_RATIO * mu
  beta_4  : [0.01, 0.1, 0.25, 0.5, 0.75, 0.9]       – β₄ ≡ 2λ₂/λ₁ ∈ (0,1)

  Derived (from v=1 and β₄ = 2λ₂/λ₁):
    λ₁ = μ² / (1 − β₄)
    λ₂ = β₄ · μ² / [2(1 − β₄)]

Fixed:
  H0      = 1.0 * mu   (change H0_RATIO to 0.3 for a second pass)
  kCutOff = [3]
  N = 512, lSide = 100, dt = 0.01, tMax = 90
"""

import os
import numpy as np

# ── Sweep axes ────────────────────────────────────────────────────────────────
MU_VALS   = [0.01]                       # physical mass scale
BETA_VALS = [0.01, 0.1, 0.25, 0.5, 0.75, 0.9]  # beta_4 = 2*lambda2/lambda1 ∈ (0,1)
KCUTOFFS  = [3]

# ── Fixed parameters ──────────────────────────────────────────────────────────
H0_RATIO = 1.0      # H0 = H0_RATIO * mu  (try 0.3 for a second scan)

N        = 512
LSIDE    = 100
DT       = 0.01
T_MAX    = 90
REMOTE_BASE = "/mt/user-batch/dpasari/scan_z4_beta4"

# ── Helpers ───────────────────────────────────────────────────────────────────
def float_str(x: float) -> str:
    """Convert float to a filename-safe string: 0.05 -> '0p05', 0.1 -> '0p1'."""
    return f"{x:g}".replace(".", "p")

def h0_label(ratio: float) -> str:
    """0.3 -> 'H0p3mu',  1.0 -> 'H1mu'."""
    r = f"{ratio:g}".replace(".", "p")
    return f"H{r}mu"

def couplings_from_mu_beta4(mu: float, beta4: float):
    """Solve {v=1, beta_4 = 2*lambda2/lambda1} for (lambda1, lambda2)."""
    assert 0.0 < beta4 < 1.0, f"Need 0 < beta_4 < 1, got {beta4}"
    lambda1 = mu * mu / (1.0 - beta4)
    lambda2 = beta4 * mu * mu / (2.0 * (1.0 - beta4))
    assert abs(lambda1) < 16 * 3.14, f"lambda1 too big: {lambda1}"
    assert abs(lambda2) < 4 * 3.14,  f"lambda2 too big: {lambda2}"
    return lambda1, lambda2

TEMPLATE = """\
#Output
outputfile = {results_dir}

#Evolution
expansion = true
evolver = LF

#Lattice
N = {N}
dt = {dt}
lSide = {lSide}

#Times
t0 = 0
tOutputFreq  = 0.1
tOutputInfreq  = 5
tOutputRareFreq = 3
tMax = {tmax}

#Spectra options
PS_type = 1
PS_version = 1

#GWs
GWprojectorType = 2
withGWs = false

fixedBackground = true
omegaEoS = 0.3333
H0 = {H0:.8f}   # {H0_ratio:.4f} * mu = {H0_ratio:.4f} * {mu}

#IC
kCutOff = {kcut}
initial_amplitudes = 0 0
initial_momenta    = 0 0

# Model Parameters  (Z4 domain-wall model, new parameterization)
# V = -mu^2 |Phi|^2 + lambda1 |Phi|^4 - lambda2 (Phi^4 + Phi*^4)
# Vacuum vev:  v = mu / sqrt(lambda1 - 2*lambda2) = 1  (unit vev)
# Constraint:  lambda1 = mu^2 + 2*lambda2
# Z4 strength: beta_4 = 2*lambda2/lambda1 = {beta:.4f}  (in (0,1))
# Check:  lambda1 - 2*lambda2 = mu^2 = {mu2:.2e}  (must equal mu^2 for v=1)
mu      = {mu}
lambda1 = {lambda1:.10f}
lambda2 = {lambda2:.10f}
"""

# ── Resolve output directory relative to notebook location ────────────────────
try:
    _root = os.path.dirname(os.path.abspath(__file__))
except NameError:
    _root = os.getcwd()  # Jupyter: cwd should be the analysis/ folder

out_dir = os.path.join(_root, "..", "src", "models", "parameter-files", "scan_z4_beta4")
os.makedirs(out_dir, exist_ok=True)

# ── Generate files ────────────────────────────────────────────────────────────
generated = []
print(f"{'File':<60} {'mu':>7} {'beta4':>7} {'kcut':>6} {'lambda1':>16} {'lambda2':>16} {'H0':>10}")
print("-" * 130)

for mu in MU_VALS:
    for beta in BETA_VALS:
        lambda1, lambda2 = couplings_from_mu_beta4(mu, beta)
        H0 = H0_RATIO * mu

        # Sanity checks
        beta_back = 2.0 * lambda2 / lambda1
        assert abs(beta_back - beta) < 1e-12, f"beta_4 mismatch: {beta_back} vs {beta}"
        vev = mu / np.sqrt(lambda1 - 2.0 * lambda2)
        assert abs(vev - 1.0) < 1e-10, f"vev = {vev} != 1 for mu={mu}, beta_4={beta}"

        mu_s = float_str(mu)
        beta_s = float_str(beta)
        hl = h0_label(H0_RATIO)

        for kcut in KCUTOFFS:
            fname   = f"DWZ4_mu{mu_s}_beta4{beta_s}_{hl}_k{kcut}.in"
            res_dir = f"{REMOTE_BASE}/results_z4_mu{mu_s}_beta4{beta_s}_{hl}_k{kcut}/"

            content = TEMPLATE.format(
                results_dir = res_dir,
                N           = N,
                dt          = DT,
                lSide       = LSIDE,
                tmax        = T_MAX,
                H0          = H0,
                H0_ratio    = H0_RATIO,
                kcut        = kcut,
                mu          = mu,
                mu2         = mu**2,
                lambda1     = lambda1,
                lambda2     = lambda2,
                beta        = beta,
            )

            fpath = os.path.join(out_dir, fname)
            with open(fpath, "w") as fh:
                fh.write(content)
            generated.append(fname)

            print(f"{fname:<60} {mu:7g} {beta:7g} {kcut:6d} {lambda1:16.6e} {lambda2:16.6e} {H0:10.4e}")

print("-" * 130)
print(f"Generated {len(generated)} files → {os.path.abspath(out_dir)}")


File                                                              mu   beta4   kcut          lambda1          lambda2         H0
----------------------------------------------------------------------------------------------------------------------------------
DWZ4_mu0p01_beta40p01_H1mu_k3.in                                0.01    0.01      3     1.010101e-04     5.050505e-07 1.0000e-02
DWZ4_mu0p01_beta40p1_H1mu_k3.in                                 0.01     0.1      3     1.111111e-04     5.555556e-06 1.0000e-02
DWZ4_mu0p01_beta40p25_H1mu_k3.in                                0.01    0.25      3     1.333333e-04     1.666667e-05 1.0000e-02
DWZ4_mu0p01_beta40p5_H1mu_k3.in                                 0.01     0.5      3     2.000000e-04     5.000000e-05 1.0000e-02
DWZ4_mu0p01_beta40p75_H1mu_k3.in                                0.01    0.75      3     4.000000e-04     1.500000e-04 1.0000e-02
DWZ4_mu0p01_beta40p9_H1mu_k3.in                                 0.01     0.9      3     1.00000

In [2]:
# ── Quick summary table ───────────────────────────────────────────────────────
# v = mu / sqrt(lambda1 - 2*lambda2) should be 1.0 for every row.
# beta_4 = 2*lambda2/lambda1 is the Z4-strength scan coordinate (in (0,1)).

import pandas as pd

rows = []
for mu in MU_VALS:
    for beta in BETA_VALS:
        lambda1, lambda2 = couplings_from_mu_beta4(mu, beta)
        H0 = H0_RATIO * mu
        vev = mu / (lambda1 - 2.0 * lambda2)**0.5
        rows.append(dict(
            mu=mu, beta_4=beta,
            lambda1=round(lambda1, 8),
            lambda2=round(lambda2, 8),
            H0=round(H0, 6),
            vev=round(vev, 8),
        ))

df = pd.DataFrame(rows)
print(df.to_string(index=False))


  mu  beta_4  lambda1  lambda2   H0  vev
0.05    0.01 0.002525 0.000013 0.05  1.0
0.05    0.10 0.002778 0.000139 0.05  1.0
0.05    0.25 0.003333 0.000417 0.05  1.0
0.05    0.50 0.005000 0.001250 0.05  1.0
0.05    0.75 0.010000 0.003750 0.05  1.0
0.05    0.90 0.025000 0.011250 0.05  1.0
0.10    0.01 0.010101 0.000051 0.10  1.0
0.10    0.10 0.011111 0.000556 0.10  1.0
0.10    0.25 0.013333 0.001667 0.10  1.0
0.10    0.50 0.020000 0.005000 0.10  1.0
0.10    0.75 0.040000 0.015000 0.10  1.0
0.10    0.90 0.100000 0.045000 0.10  1.0
